# 🚀 3D Perception & Grasping Server (Orbbec Gemini E + Grounded-SAM on Kaggle GPU)

This notebook runs the **Perception & 6-DoF Robotic Grasping Server** using Kaggle's free GPU (Tesla T4 / P100) and **Cloudflare Tunnels (Zero setup / No tokens required)**.

### How It Works:
1. **Your Local PC** has the **Orbbec Gemini E** camera plugged in via USB.
2. **This Kaggle Notebook** loads Grounding DINO + SAM on **CUDA GPU**.
3. **Cloudflare Tunnel (`cloudflared`)** exposes the Kaggle server via a secure public `https://*.trycloudflare.com` URL.
4. Local PC streams RGB + Depth $\rightarrow$ Kaggle GPU computes 3D coordinates & 6-DoF grasp poses $\rightarrow$ Results displayed in real time!

In [ ]:
# Step 1: Verify Kaggle GPU
!nvidia-smi

In [ ]:
# Step 2: Install Required Dependencies
!pip install -q fastapi uvicorn websockets opencv-python transformers timm supervision safetensors open3d

In [ ]:
# Step 3: Install Cloudflare Tunnel Client (cloudflared)
# No account, token, or credit card required!
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
!cloudflared --version

In [ ]:
# Step 4: Clone Your GitHub Repository
# REPLACE WITH YOUR GITHUB REPO URL:
GIT_REPO_URL = "https://github.com/your-username/Segmentation_based_vision.git"

import os
repo_name = GIT_REPO_URL.split("/")[-1].replace(".git", "")
if not os.path.exists(repo_name):
    !git clone $GIT_REPO_URL

%cd $repo_name

In [ ]:

# Step 5: Start Cloudflare Tunnel in Background (Kaggle-compatible)

import subprocess
import time
import re
import os
from pathlib import Path

log_path = Path("/kaggle/working/cloudflared.log")
log_file = open(log_path, "w")

# Start cloudflared without using shell background syntax (&)
cloudflared_process = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url", "http://127.0.0.1:8000",
        "--metrics", "127.0.0.1:45678",
        "--no-autoupdate",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

print("[*] Starting Cloudflare Tunnel...")

tunnel_url = None

for attempt in range(30):
    time.sleep(1)

    # Detect an early process failure
    if cloudflared_process.poll() is not None:
        print("[!] cloudflared exited unexpectedly.")
        break

    content = log_path.read_text(errors="replace")

    matches = re.findall(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        content,
    )

    if matches:
        tunnel_url = matches[0]
        break

if tunnel_url:
    print("=" * 65)
    print("CLOUDFLARE PUBLIC TUNNEL URL:")
    print(tunnel_url)
    print()
    print("Use this base URL in your local client_streamer.py.")
    print("FastAPI must be running on port 8000.")
    print("=" * 65)
else:
    print("[!] Tunnel URL not detected.")
    print("Check the log below:")
    print(log_path.read_text(errors="replace")[-5000:])
    print("\nProcess status:", cloudflared_process.poll())


In [ ]:
# Step 6: Start Kaggle GPU Perception & Grasping Server
!python kaggle_server.py